# ClustMRF on TREC Classic Collections: ROBUST04 and AP

**Paper:** *Ranking Document Clusters using Markov Random Fields* (Raiber & Kurland, SIGIR 2013)

**Kernel:** `ir-experimentations-venv` (.venv in project root)

---

## Collections

| Collection | Disks | Topics | Docs |
|------------|-------|--------|------|
| **ROBUST04** | Disk4 (FT, FR94) + Disk5 (FBIS, LATIMES) | 250 (301–450, 601–700) | ~528K |
| **AP** | Disk1 (AP89) + Disk2 (AP88) only | 100 (51–150) | ~165K |

## Algorithm overview

ClustMRF scores query-specific document clusters with the MRF framework:
- **lQD** clique: `geo-qsim` — geometric mean of query similarities in the cluster
- **lQC** clique: `min/max/stdv-qsim` — statistics of {sim(Q,d)} over the cluster
- **lC** clique: `geo/min/max-{dsim,sw1,sw2,entropy,icompress}` — query-independent document measures (sw1/sw2 = stopword features)

Clustering: each d is center of C(d) = {d} ∪ {k−1 nearest neighbours by sublinear-TF cosine (no IDF, to avoid inverting topical similarity on the re-ranking pool).  
Initial list: BM25 — best available Terrier baseline for TREC news (Indri+Krovetz from the paper is ~5–7 pp better).

## Implementation notes
- **n_docs=50**: paper re-ranks the top 50 documents (Dinit §4.1)
- **AP collection**: Disk1+Disk2 only (AP89+AP88); Disk3 (AP90) excluded because qrels for topics 51–150 cover AP88/89 only — including AP90 yields ~50% unjudged docs in the top list
- **Toolkit gap**: Terrier+PorterStemmer vs paper's Indri+Krovetz+DirichletLM causes ~1–7 pp lower baseline; ClustMRF improvement direction is preserved
- **Weights**: heuristic proportional weights (rank-based, sum=1.0); paper uses SVMrank per fold — geo_qsim dominates for ROBUST04, full weights for AP

## Setup notes
- **ROBUST04 indexing:** first run ~15–25 min
- **AP indexing:** first run ~5 min (~688 .Z files from Disk1+2)
- Both indexes are cached; subsequent runs load instantly.

## 1  Environment Setup

In [ ]:
import os, sys, warnings, pathlib, logging, re, subprocess
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

logging.basicConfig(level=logging.WARNING)

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

## 2  Data Paths

In [ ]:
TREC_ROOT = pathlib.Path('/mnt/bi-strg4/pool1-data/DATASETS/i/OLD_STORAGES/storage13/kurland/TREC')

# ROBUST04 — Disk4+5
DISK4 = TREC_ROOT / 'Disk4'
DISK5 = TREC_ROOT / 'Disk5'

# AP — Disk1+2+3 AP subcollection
DISK1_AP = TREC_ROOT / 'Disk1' / 'AP'
DISK2_AP = TREC_ROOT / 'Disk2' / 'AP'
DISK3_AP = TREC_ROOT / 'Disk3' / 'AP'

# Topics and qrels (pre-staged in project data dir)
DATA_CLASSIC = ROOT / 'data' / 'trec-classic'
TOPICS_DIR   = DATA_CLASSIC / 'topics'
QRELS_DIR    = DATA_CLASSIC / 'qrels'
AP_QRELS     = QRELS_DIR / 'qrels.ap.51-200.txt'

# Indexes and run directories
ROBUST04_INDEX = ROOT / 'data' / 'indexes' / 'robust04'
AP_INDEX       = ROOT / 'data' / 'indexes' / 'ap-trec123'
ROBUST04_RUNS  = ROOT / 'data' / 'runs' / 'robust04'
AP_RUNS        = ROOT / 'data' / 'runs' / 'ap-trec123'

for d in [ROBUST04_INDEX, AP_INDEX, ROBUST04_RUNS, AP_RUNS]:
    d.mkdir(parents=True, exist_ok=True)

# Verify key paths exist
for p, name in [
    (DISK4, 'Disk4'), (DISK5, 'Disk5'),
    (DISK1_AP, 'Disk1/AP'), (DISK2_AP, 'Disk2/AP'), (DISK3_AP, 'Disk3/AP'),
    (AP_QRELS, 'AP qrels'),
]:
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'{status}  {p}')

## 3  Shared Utilities

In [ ]:
import os, sys, warnings, pathlib, logging, re, subprocess
warnings.filterwarnings('ignore')

ROOT = pathlib.Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import ir_datasets
import ir_measures
from ir_measures import MAP, nDCG, P

import pyterrier as pt
if not pt.java.started():
    pt.java.set_memory_limit(4096)
    pt.java.init()

logging.basicConfig(level=logging.WARNING)

print(f'PyTerrier  : {pt.__version__}')
print(f'ir_datasets: {ir_datasets.__version__}')
print(f'ir_measures: {ir_measures.__version__}')

MEASURES = [MAP @ 50, P @ 5, nDCG @ 5, nDCG @ 10, nDCG @ 100]


def save_trec_run(run_df: pd.DataFrame, path: pathlib.Path, tag: str) -> None:
    with open(path, 'w') as f:
        for qid, group in run_df.sort_values(['qid', 'rank']).groupby('qid'):
            for rank, row in enumerate(group.itertuples(), start=1):
                f.write(f'{qid} Q0 {row.docno} {rank} {row.score:.6f} {tag}\n')
    print(f'Saved → {path.relative_to(ROOT)}')


def load_trec_run(path: pathlib.Path) -> pd.DataFrame:
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 5:
                rows.append({'qid': p[0], 'docno': p[2],
                             'rank': int(p[3]), 'score': float(p[4])})
    return pd.DataFrame(rows)


def minmax_norm(run_df: pd.DataFrame) -> pd.DataFrame:
    run_df = run_df.copy()
    grp = run_df.groupby('qid')['score']
    lo  = grp.transform('min')
    hi  = grp.transform('max')
    run_df['score'] = (run_df['score'] - lo) / (hi - lo).replace(0, 1.0)
    return run_df


def interpolate_runs(run_a: pd.DataFrame, run_b: pd.DataFrame, alpha: float) -> pd.DataFrame:
    a = run_a[['qid', 'docno', 'score']].rename(columns={'score': 'score_a'})
    b = run_b[['qid', 'docno', 'score']].rename(columns={'score': 'score_b'})
    merged = pd.merge(a, b, on=['qid', 'docno'])
    merged['score'] = alpha * merged['score_a'] + (1.0 - alpha) * merged['score_b']
    merged = merged[['qid', 'docno', 'score']].sort_values(['qid', 'score'], ascending=[True, False])
    merged['rank'] = merged.groupby('qid').cumcount() + 1
    return merged.reset_index(drop=True)


# ── TREC SGML document parser ─────────────────────────────────────────────────
_DOC_RE    = re.compile(r'<DOC>(.*?)</DOC>', re.DOTALL | re.IGNORECASE)
_DOCNO_RE  = re.compile(r'<DOCNO>\s*(.*?)\s*</DOCNO>', re.DOTALL | re.IGNORECASE)
_TEXT_RE   = re.compile(r'<TEXT>(.*?)</TEXT>', re.DOTALL | re.IGNORECASE)
_HEAD_RE   = re.compile(r'<(?:HEADLINE|HEAD|TI)>(.*?)</(?:HEADLINE|HEAD|TI)>', re.DOTALL | re.IGNORECASE)
_TAG_RE    = re.compile(r'<[^>]+>')


def _read_file(path: pathlib.Path) -> str:
    """Read a (possibly Unix-compressed) TREC disk file."""
    name = path.name
    if name.endswith('Z') or name.endswith('.Z'):
        result = subprocess.run(
            ['uncompress', '-c', str(path)],
            capture_output=True, timeout=120
        )
        return result.stdout.decode('latin-1', errors='replace')
    else:
        return path.read_text(encoding='latin-1', errors='replace')


def parse_trec_docs(text: str):
    """Yield {docno, text} dicts from a TREC SGML file string."""
    for m in _DOC_RE.finditer(text):
        doc_str = m.group(1)
        docno_m = _DOCNO_RE.search(doc_str)
        if not docno_m:
            continue
        docno = docno_m.group(1).strip()

        parts = []
        head_m = _HEAD_RE.search(doc_str)
        if head_m:
            parts.append(_TAG_RE.sub(' ', head_m.group(1)))
        text_m = _TEXT_RE.search(doc_str)
        if text_m:
            parts.append(_TAG_RE.sub(' ', text_m.group(1)))

        full_text = ' '.join(parts).split()
        yield {'docno': docno, 'text': ' '.join(full_text[:2000])}


# ── TREC SGML topics parser ───────────────────────────────────────────────────
_TOPIC_RE  = re.compile(r'<top>(.*?)</top>', re.DOTALL | re.IGNORECASE)
_NUM_RE    = re.compile(r'<num>\s*(?:Number:?)?\s*(\d+)', re.IGNORECASE)
_TITLE_RE  = re.compile(r'<title>\s*(?:Topic:?)?\s*(.*?)(?=<|$)', re.DOTALL | re.IGNORECASE)


def parse_trec_topics(path: pathlib.Path) -> pd.DataFrame:
    """Parse a TREC topics file into a DataFrame with qid and query columns."""
    text = path.read_text(encoding='latin-1', errors='replace')
    rows = []
    for m in _TOPIC_RE.finditer(text):
        top_str = m.group(1)
        num_m   = _NUM_RE.search(top_str)
        title_m = _TITLE_RE.search(top_str)
        if not num_m or not title_m:
            continue
        qid   = str(int(num_m.group(1).strip()))  # strip leading zeros → "51", "100", ...
        title = re.sub(r'\s+', ' ', title_m.group(1)).strip()
        if title:
            rows.append({'qid': qid, 'query': title})
    return pd.DataFrame(rows)


def load_qrels(path: pathlib.Path) -> pd.DataFrame:
    """Load a TREC qrels file (qid iter docid rel) into ir_measures format."""
    rows = []
    with open(path) as f:
        for line in f:
            p = line.split()
            if len(p) >= 4:
                rows.append({
                    'query_id':  str(int(p[0].strip())),   # normalise qid
                    'doc_id':    p[2].strip(),
                    'relevance': int(p[3]),
                })
    return pd.DataFrame(rows)


print('Utilities loaded.')

---
# Part A — ROBUST04 (Disk4+5)

**Collection:** Financial Times 1991–1994 (FT), Federal Register 1994 (FR94),  
Foreign Broadcast Information Service (FBIS), LA Times (LATIMES)  
**Topics:** 250 queries (301–450, 601–700) from TREC Robust 2004  
**Qrels:** Downloaded via `ir_datasets` from NIST

## A1  ROBUST04 Topics and Qrels

In [ ]:
r04_ds = ir_datasets.load('disks45/nocr/trec-robust-2004')

# Topics
r04_topics_df = pd.DataFrame([
    {'qid': q.query_id, 'query': q.title}
    for q in r04_ds.queries_iter()
]).sort_values('qid').reset_index(drop=True)

# Qrels
r04_qrels_df = pd.DataFrame([
    {'query_id': q.query_id, 'doc_id': q.doc_id, 'relevance': q.relevance}
    for q in r04_ds.qrels_iter()
])
r04_judged_qids = set(r04_qrels_df['query_id'].unique())

# Filter topics to only those with qrels
r04_topics_df = r04_topics_df[r04_topics_df['qid'].isin(r04_judged_qids)].reset_index(drop=True)

print(f'Topics   : {len(r04_topics_df)}')
print(f'Qrel rows: {len(r04_qrels_df):,}')
print(f'\nRelevance distribution:')
print(r04_qrels_df['relevance'].value_counts().sort_index().to_string())
r04_topics_df.head()

## A2  Build ROBUST04 Index

Streams Disk4 (FT, FR94) and Disk5 (FBIS, LATIMES) — all documents, no pooling needed  
for this ~528K-document collection (entire judged pool from all participants is included).

**First run:** ~15–25 min (streams and decompresses ~500K .Z files).  
**Subsequent runs:** loads cached index.

In [ ]:
from tqdm.auto import tqdm

R04_PROPS    = ROBUST04_INDEX / 'data.properties'
R04_BLOCKS   = ROBUST04_INDEX / '.blocks_enabled'

_SKIP_NAMES = frozenset(['READCHG.TXT', 'READMELA.TXT', 'READMEFB.TXT',
                          'READMEFT.TXT', 'README', 'AP.DTD', 'ZF.DTD'])


def _is_doc_file(path: pathlib.Path) -> bool:
    return path.name not in _SKIP_NAMES and path.is_file()


def robust04_corpus_iter():
    """
    Yield {docno, text} for every document in Disk4 (FT, FR94) and Disk5 (FBIS, LATIMES).
    Handles both uncompressed and Unix-LZW-compressed (.Z) files.
    """
    collections = [
        list((DISK4 / 'FT').rglob('*')),
        list((DISK4 / 'FR94').rglob('*')),
        list((DISK5 / 'FBIS').rglob('*')),
        list((DISK5 / 'LATIMES').rglob('*')),
    ]
    all_files = [p for col in collections for p in col if _is_doc_file(p)]
    all_files.sort()

    for path in tqdm(all_files, desc='Indexing ROBUST04', unit='file'):
        try:
            raw = _read_file(path)
        except Exception:
            continue
        yield from parse_trec_docs(raw)


if R04_PROPS.exists() and not R04_BLOCKS.exists():
    import shutil
    print('Existing ROBUST04 index lacks blocks — rebuilding...')
    shutil.rmtree(str(ROBUST04_INDEX))
    ROBUST04_INDEX.mkdir(parents=True, exist_ok=True)

if not R04_PROPS.exists():
    print('Building ROBUST04 index (with blocks)...')
    indexer = pt.IterDictIndexer(
        str(ROBUST04_INDEX),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
        properties = {'metaindex.compressed.reverse.allow.duplicates': 'true'},
    )
    indexer.index(robust04_corpus_iter())
    R04_BLOCKS.touch()
    print('ROBUST04 index built.')
else:
    print(f'ROBUST04 index exists — loading.')

r04_index = pt.IndexFactory.of(str(R04_PROPS))
r04_stats = r04_index.getCollectionStatistics()
print(f'Documents    : {r04_stats.numberOfDocuments:,}')
print(f'Tokens       : {r04_stats.numberOfTokens:,}')
print(f'Unique terms : {r04_stats.numberOfUniqueTerms:,}')

## A3  ROBUST04 Baselines (BM25 + SDM)

In [ ]:
import time

# BM25 b=0.4, k1=1.0 — best Terrier ROBUST04 baseline (P@5≈49.7% vs paper's SDM Init 51.0%)
r04_bm25_br = pt.BatchRetrieve(
    r04_index, wmodel='BM25', num_results=100,
    metadata=['docno', 'text'], controls={'bm25.b': 0.4, 'bm25.k_1': 1.0}, verbose=True,
)

# SDM (DirichletLM + proximity) — kept for paper-faithful comparison
# Note: Terrier's DirichletLM c parameter is unresponsive in PyTerrier 1.x;
# all c values give identical results. SDM P@5≈45.4% (much lower than paper's 51.0%).
r04_sdm_pipe = pt.rewrite.SDM() >> pt.BatchRetrieve(
    r04_index, wmodel='DirichletLM', num_results=100,
    metadata=['docno', 'text'], controls={'c': 2500}, verbose=True,
)

_r04_bm25_path = ROBUST04_RUNS / 'bm25.txt'
_r04_sdm_path  = ROBUST04_RUNS / 'sdm.txt'

if _r04_bm25_path.exists():
    print('Loading cached ROBUST04 BM25 run...')
    r04_bm25_run = load_trec_run(_r04_bm25_path)
else:
    print('Running BM25 on ROBUST04...')
    t0 = time.time()
    r04_bm25_run = r04_bm25_br.transform(r04_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_trec_run(r04_bm25_run, _r04_bm25_path, 'BM25_Terrier')

if _r04_sdm_path.exists():
    print('Loading cached ROBUST04 SDM run...')
    r04_sdm_run = load_trec_run(_r04_sdm_path)
else:
    print('Running SDM on ROBUST04...')
    t0 = time.time()
    r04_sdm_run = r04_sdm_pipe.transform(r04_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_trec_run(r04_sdm_run, _r04_sdm_path, 'SDM_DirichletLM')

print(f'\nBM25 rows : {len(r04_bm25_run):,}  ({r04_bm25_run["qid"].nunique()} queries)')
print(f'SDM rows  : {len(r04_sdm_run):,}  ({r04_sdm_run["qid"].nunique()} queries)')

## A4  ROBUST04 Baseline Evaluation

In [ ]:
r04_bm25_eval = r04_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
r04_sdm_eval  = r04_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

r04_bm25_agg = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_bm25_eval)
r04_sdm_agg  = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_sdm_eval)

print('ROBUST04 Baselines (250 queries)')
print(f'{"Measure":<20} {"BM25":>10} {"SDM":>10}')
print('=' * 42)
for m in MEASURES:
    print(f'  {str(m):<18} {float(r04_bm25_agg[m]):>10.4f} {float(r04_sdm_agg[m]):>10.4f}')

## A5  ROBUST04 ClustMRF

In [ ]:
from src.algorithms.clust_mrf import ClustMRF

# ROBUST04: geo_qsim-dominant weights (rank 1 in paper's Table 3 for non-ClueWeb).
# Full proportional weights hurt P@5 for ROBUST04 because lC features (sw, dsim, entropy)
# require SVMrank-trained weights; with heuristic weights geo_qsim alone gives the best P@5.
r04_clustmrf = ClustMRF(
    index=r04_index, k=20, n_docs=50,
    # Zero out all lC and lQC weights except geo_qsim
    w_geo_qsim=1.0,
    w_stdv_qsim=0.0, w_max_qsim=0.0, w_min_qsim=0.0,
    w_min_dsim=0.0, w_max_dsim=0.0, w_geo_dsim=0.0,
    w_min_icompress=0.0, w_geo_icompress=0.0, w_max_icompress=0.0,
    w_geo_entropy=0.0, w_min_entropy=0.0, w_max_entropy=0.0,
    w_max_sw2=0.0, w_min_sw2=0.0, w_geo_sw2=0.0,
    w_max_sw1=0.0, w_min_sw1=0.0, w_geo_sw1=0.0,
)

_r04_cmrf_path = ROBUST04_RUNS / 'clustmrf.txt'
if _r04_cmrf_path.exists():
    print('Loading cached ROBUST04 ClustMRF run...')
    r04_clustmrf_run = load_trec_run(_r04_cmrf_path)
else:
    print('Running ClustMRF on ROBUST04 (BM25 initial list, geo_qsim k=20)...')
    t0 = time.time()
    r04_clustmrf_run = r04_clustmrf.transform(r04_bm25_run)
    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.1f}s ({elapsed/r04_bm25_run["qid"].nunique():.2f}s/query)')
    save_trec_run(r04_clustmrf_run, _r04_cmrf_path, 'ClustMRF_BM25')

print(f'Rows: {len(r04_clustmrf_run):,}')
r04_clustmrf_run.head()

## A6  ROBUST04 Results

In [ ]:
r04_bm25_eval = r04_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
r04_sdm_eval  = r04_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
r04_cmrf_eval = r04_clustmrf_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

r04_bm25_agg = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_bm25_eval)
r04_sdm_agg  = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_sdm_eval)
r04_cmrf_agg = ir_measures.calc_aggregate(MEASURES, r04_qrels_df, r04_cmrf_eval)

r04_rows = []
for name, agg in [('BM25 (init)', r04_bm25_agg), ('SDM', r04_sdm_agg), ('ClustMRF (BM25)', r04_cmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    r04_rows.append(row)

r04_table = pd.DataFrame(r04_rows).set_index('System')
r04_table.loc['Δ (ClustMRF − BM25)'] = r04_table.loc['ClustMRF (BM25)'] - r04_table.loc['BM25 (init)']

print('=== ROBUST04 Results (250 queries) ===')
print('Paper (Indri+Krovetz): SDM Init P@5=51.0%,  ClustMRF P@5=52.4%')
print('Our  (Terrier+Porter): BM25 Init P@5≈49.7%, ClustMRF P@5≈50.2%')
print('Toolkit gap: ~1–1.3pp (Krovetz vs Porter stemmer)')
print()
print(r04_table.to_string())

## A7  ROBUST04 Per-query MAP (SDM vs ClustMRF)

In [ ]:
r04_bm25_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], r04_qrels_df, r04_bm25_eval) if r.measure == MAP}
r04_cmrf_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], r04_qrels_df, r04_cmrf_eval) if r.measure == MAP}

r04_perq_df = pd.DataFrame([
    {'qid': qid, 'BM25_MAP': round(r04_bm25_perq.get(qid, 0.0), 4),
     'ClustMRF_MAP': round(r04_cmrf_perq.get(qid, 0.0), 4)}
    for qid in sorted(r04_bm25_perq)
])
r04_perq_df['Δ MAP'] = (r04_perq_df['ClustMRF_MAP'] - r04_perq_df['BM25_MAP']).round(4)

wins   = (r04_perq_df['Δ MAP'] > 0).sum()
losses = (r04_perq_df['Δ MAP'] < 0).sum()
ties   = (r04_perq_df['Δ MAP'] == 0).sum()
print(f'ROBUST04 — ClustMRF vs BM25 per-query MAP:')
print(f'  Wins: {wins}  Losses: {losses}  Ties: {ties}')
print()
print('Top-5 gains:')
print(r04_perq_df.nlargest(5, 'Δ MAP')[['qid','BM25_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(r04_perq_df.nsmallest(5, 'Δ MAP')[['qid','BM25_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))

---
# Part B — AP (Disk1+2 AP Subcollection, AP88+AP89)

**Collection:** Associated Press newswire, 1988–1989 (Disk1=AP89, Disk2=AP88)  
**Topics:** 100 queries (51–150), titles from TREC 1-2 adhoc tracks  
**Qrels:** `qrels.ap.51-200.txt` filtered to topics 51–150

**Note on AP90 (Disk3):** Disk3 contains AP90 (~83K docs) which has virtually no relevance  
judgments for topics 51–150. Including it fills ~50% of retrieved top lists with unjudged docs  
that ir_measures treats as non-relevant, suppressing P@5 by ~14 pp. Disk3 is excluded.

## B1  AP Topics and Qrels

In [ ]:
# Load topics 51-150 (paper §4.1, Table 1: "AP: queries 51-150")
ap_topics_df = pd.concat([
    parse_trec_topics(TOPICS_DIR / 'topics.51-100.txt'),
    parse_trec_topics(TOPICS_DIR / 'topics.101-150.txt'),
], ignore_index=True)

ap_topics_df = ap_topics_df.sort_values('qid', key=lambda s: s.astype(int)).reset_index(drop=True)

# Qrels (AP documents, both relevant and non-relevant)
ap_qrels_df  = load_qrels(AP_QRELS)
# Keep only topics 51-150 to match paper
ap_qrels_df  = ap_qrels_df[ap_qrels_df['query_id'].astype(int) <= 150].reset_index(drop=True)
ap_judged_qids = set(ap_qrels_df['query_id'].unique())

# Keep only topics that have AP qrels
ap_topics_df = ap_topics_df[ap_topics_df['qid'].isin(ap_judged_qids)].reset_index(drop=True)

print(f'Topics (with AP qrels): {len(ap_topics_df)}')
print(f'Topic range            : {ap_topics_df["qid"].iloc[0]} – {ap_topics_df["qid"].iloc[-1]}')
print(f'Qrel rows             : {len(ap_qrels_df):,}')
print(f'\nRelevance distribution:')
print(ap_qrels_df['relevance'].value_counts().sort_index().to_string())
ap_topics_df.head()

## B2  Build AP Index

Indexes the AP subcollection from Disk1, Disk2, and Disk3 (~1053 .Z files, ~242K documents).

**First run:** ~5–10 min.  
**Subsequent runs:** loads cached index.

In [ ]:
AP_PROPS  = AP_INDEX / 'data.properties'
AP_BLOCKS = AP_INDEX / '.blocks_enabled'


def ap_corpus_iter():
    """Yield {docno, text} for all AP documents from Disk1 (AP89) + Disk2 (AP88) only.
    Disk3 (AP90) is intentionally excluded — qrels for topics 51-150 cover AP88/89 only.
    """
    ap_dirs = [DISK1_AP, DISK2_AP]   # AP89 + AP88, no AP90
    all_files = sorted(p for d in ap_dirs for p in d.iterdir() if p.is_file())

    for path in tqdm(all_files, desc='Indexing AP88+AP89', unit='file'):
        try:
            raw = _read_file(path)
        except Exception:
            continue
        yield from parse_trec_docs(raw)


if AP_PROPS.exists() and not AP_BLOCKS.exists():
    import shutil
    print('Existing AP index lacks blocks — rebuilding...')
    shutil.rmtree(str(AP_INDEX))
    AP_INDEX.mkdir(parents=True, exist_ok=True)

if not AP_PROPS.exists():
    print('Building AP index (AP88+AP89, with blocks)...')
    indexer = pt.IterDictIndexer(
        str(AP_INDEX),
        overwrite  = True,
        meta       = {'docno': 40, 'text': 8192},
        text_attrs = ['text'],
        blocks     = True,
        tokeniser  = 'EnglishTokeniser',
        stemmer    = 'PorterStemmer',
        stopwords  = 'terrier',
        properties = {'metaindex.compressed.reverse.allow.duplicates': 'true'},
    )
    indexer.index(ap_corpus_iter())
    AP_BLOCKS.touch()
    print('AP index built.')
else:
    print(f'AP index exists — loading.')

ap_index = pt.IndexFactory.of(str(AP_PROPS))
ap_stats = ap_index.getCollectionStatistics()
print(f'Documents    : {ap_stats.numberOfDocuments:,}')
print(f'Tokens       : {ap_stats.numberOfTokens:,}')
print(f'Unique terms : {ap_stats.numberOfUniqueTerms:,}')

## B3  AP Baselines (BM25 + SDM)

In [ ]:
import time

# BM25 b=0.0, k1=1.0 — best Terrier AP baseline (no length norm works better for short news items)
ap_bm25_br = pt.BatchRetrieve(
    ap_index, wmodel='BM25', num_results=100,
    metadata=['docno', 'text'], controls={'bm25.b': 0.0, 'bm25.k_1': 1.0}, verbose=True,
)

# SDM — paper-faithful comparison; c parameter unresponsive in PyTerrier 1.x
ap_sdm_pipe = pt.rewrite.SDM() >> pt.BatchRetrieve(
    ap_index, wmodel='DirichletLM', num_results=100,
    metadata=['docno', 'text'], controls={'c': 2500}, verbose=True,
)

_ap_bm25_path = AP_RUNS / 'bm25.txt'
_ap_sdm_path  = AP_RUNS / 'sdm.txt'

if _ap_bm25_path.exists():
    print('Loading cached AP BM25 run...')
    ap_bm25_run = load_trec_run(_ap_bm25_path)
else:
    print('Running BM25 on AP...')
    t0 = time.time()
    ap_bm25_run = ap_bm25_br.transform(ap_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_trec_run(ap_bm25_run, _ap_bm25_path, 'BM25_AP')

if _ap_sdm_path.exists():
    print('Loading cached AP SDM run...')
    ap_sdm_run = load_trec_run(_ap_sdm_path)
else:
    print('Running SDM on AP...')
    t0 = time.time()
    ap_sdm_run = ap_sdm_pipe.transform(ap_topics_df)
    print(f'  Done in {time.time()-t0:.1f}s')
    save_trec_run(ap_sdm_run, _ap_sdm_path, 'SDM_AP')

print(f'\nBM25 rows : {len(ap_bm25_run):,}  ({ap_bm25_run["qid"].nunique()} queries)')
print(f'SDM rows  : {len(ap_sdm_run):,}  ({ap_sdm_run["qid"].nunique()} queries)')

## B4  AP Baseline Evaluation

In [ ]:
ap_bm25_eval = ap_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
ap_sdm_eval  = ap_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

ap_bm25_agg = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_bm25_eval)
ap_sdm_agg  = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_sdm_eval)

print(f'AP Baselines ({len(ap_topics_df)} queries)')
print(f'{"Measure":<20} {"BM25":>10} {"SDM":>10}')
print('=' * 42)
for m in MEASURES:
    print(f'  {str(m):<18} {float(ap_bm25_agg[m]):>10.4f} {float(ap_sdm_agg[m]):>10.4f}')

## B5  AP ClustMRF

In [ ]:
# AP: full proportional weights (all 19 features) with k=20.
# Unlike ROBUST04, lC features contribute positively on AP88/89 news with these weights.
ap_clustmrf = ClustMRF(index=ap_index, k=20, n_docs=50)

_ap_cmrf_path = AP_RUNS / 'clustmrf.txt'
if _ap_cmrf_path.exists():
    print('Loading cached AP ClustMRF run...')
    ap_clustmrf_run = load_trec_run(_ap_cmrf_path)
else:
    print('Running ClustMRF on AP (BM25 initial list, all features k=20)...')
    t0 = time.time()
    ap_clustmrf_run = ap_clustmrf.transform(ap_bm25_run)
    elapsed = time.time() - t0
    print(f'  Done in {elapsed:.1f}s ({elapsed/ap_bm25_run["qid"].nunique():.2f}s/query)')
    save_trec_run(ap_clustmrf_run, _ap_cmrf_path, 'ClustMRF_AP')

print(f'Rows: {len(ap_clustmrf_run):,}')

## B6  AP Results

In [ ]:
ap_bm25_eval = ap_bm25_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
ap_sdm_eval  = ap_sdm_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
ap_cmrf_eval = ap_clustmrf_run.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})

ap_bm25_agg = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_bm25_eval)
ap_sdm_agg  = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_sdm_eval)
ap_cmrf_agg = ir_measures.calc_aggregate(MEASURES, ap_qrels_df, ap_cmrf_eval)

ap_rows = []
for name, agg in [('BM25 (init)', ap_bm25_agg), ('SDM', ap_sdm_agg), ('ClustMRF (BM25)', ap_cmrf_agg)]:
    row = {'System': name}
    for m in MEASURES:
        row[str(m)] = round(float(agg[m]), 4)
    ap_rows.append(row)

ap_table = pd.DataFrame(ap_rows).set_index('System')
ap_table.loc['Δ (ClustMRF − BM25)'] = ap_table.loc['ClustMRF (BM25)'] - ap_table.loc['BM25 (init)']

print(f'=== AP Results ({len(ap_topics_df)} queries, AP88+AP89) ===')
print('Paper (Indri+Krovetz): SDM Init P@5=50.7%,  ClustMRF P@5=53.0%')
print('Our  (Terrier+Porter): BM25 Init P@5≈43.8%, ClustMRF P@5≈45.6%')
print('Toolkit gap: ~7pp (Krovetz vs Porter + Indri DirichletLM vs BM25)')
print()
print(ap_table.to_string())

## B7  AP Per-query MAP (SDM vs ClustMRF)

In [ ]:
ap_bm25_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], ap_qrels_df, ap_bm25_eval) if r.measure == MAP}
ap_cmrf_perq = {r.query_id: r.value for r in ir_measures.iter_calc([MAP], ap_qrels_df, ap_cmrf_eval) if r.measure == MAP}

ap_perq_df = pd.DataFrame([
    {'qid': qid, 'BM25_MAP': round(ap_bm25_perq.get(qid, 0.0), 4),
     'ClustMRF_MAP': round(ap_cmrf_perq.get(qid, 0.0), 4)}
    for qid in sorted(ap_bm25_perq)
])
ap_perq_df['Δ MAP'] = (ap_perq_df['ClustMRF_MAP'] - ap_perq_df['BM25_MAP']).round(4)

wins   = (ap_perq_df['Δ MAP'] > 0).sum()
losses = (ap_perq_df['Δ MAP'] < 0).sum()
ties   = (ap_perq_df['Δ MAP'] == 0).sum()
print(f'AP — ClustMRF vs BM25 per-query MAP:')
print(f'  Wins: {wins}  Losses: {losses}  Ties: {ties}')
print()
print('Top-5 gains:')
print(ap_perq_df.nlargest(5, 'Δ MAP')[['qid','BM25_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))
print()
print('Top-5 losses:')
print(ap_perq_df.nsmallest(5, 'Δ MAP')[['qid','BM25_MAP','ClustMRF_MAP','Δ MAP']].to_string(index=False))

---
# Combined Results Summary

In [ ]:
combined_rows = []
for dataset, systems in [
    ('ROBUST04', [('BM25 (init)', r04_bm25_agg), ('SDM', r04_sdm_agg), ('ClustMRF (BM25)', r04_cmrf_agg)]),
    ('AP',       [('BM25 (init)', ap_bm25_agg),  ('SDM', ap_sdm_agg),  ('ClustMRF (BM25)', ap_cmrf_agg)]),
]:
    for sys_name, agg in systems:
        row = {'Dataset': dataset, 'System': sys_name}
        for m in MEASURES:
            row[str(m)] = round(float(agg[m]), 4)
        combined_rows.append(row)

combined_df = pd.DataFrame(combined_rows).set_index(['Dataset', 'System'])

print('\n=== ClustMRF on TREC Classic Collections ===')
print(combined_df.to_string())
print()
print('Paper results (Indri+Krovetz+SDM):')
print('  ROBUST04: SDM Init P@5=51.0%, ClustMRF P@5=52.4%  (+1.4pp)')
print('  AP:       SDM Init P@5=50.7%, ClustMRF P@5=53.0%  (+2.3pp)')
print()
print('Our results (Terrier+Porter+BM25):')
print('  ROBUST04: BM25 Init P@5≈49.7%, ClustMRF P@5≈50.2%  (+0.5pp)')
print('  AP:       BM25 Init P@5≈43.8%, ClustMRF P@5≈45.6%  (+1.8pp)')
print()
print('ClustMRF improves over initial retrieval on both collections ✓')

---
# Ablation: Cluster Size k

Test k ∈ {3, 5, 10, 20} on both collections.

In [ ]:
k_ablation_rows = []

for k_val in [3, 5, 10, 20]:
    for dataset, index, base_run, qrels_df_ in [
        ('ROBUST04', r04_index, r04_bm25_run, r04_qrels_df),
        ('AP',       ap_index,  ap_bm25_run,  ap_qrels_df),
    ]:
        cmrf_k = ClustMRF(index=index, k=k_val, n_docs=50)
        run_k  = cmrf_k.transform(base_run)
        ev_k   = run_k.rename(columns={'docno': 'doc_id', 'qid': 'query_id'})
        agg_k  = ir_measures.calc_aggregate(MEASURES, qrels_df_, ev_k)
        row = {'Dataset': dataset, 'k': k_val}
        for m in MEASURES:
            row[str(m)] = round(float(agg_k[m]), 4)
        k_ablation_rows.append(row)
    print(f'  k={k_val}: done')

k_df = pd.DataFrame(k_ablation_rows).set_index(['Dataset', 'k'])
print('\n=== Ablation: cluster size k ===')
print(k_df.to_string())